# Embeddings — omarelshehy/Arabic-Retrieval-v1.0

توحيد التشكيل قبل الترميز (الوثائق مشكّلة والاستعلامات لا) + تقييم mini-eval مسجّل النواتج.

In [1]:
import json
import re

import numpy as np
from sentence_transformers import SentenceTransformer

with open("../data/documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

print("Number of documents:", len(documents))

Number of documents: 116


In [2]:
import os

from dotenv import find_dotenv, load_dotenv
from huggingface_hub import login

# توكن HuggingFace من .env فقط (لا يكتب في الكود)
load_dotenv(find_dotenv())
login(token=os.environ["huggingface_Access_Tokens"])

# توحيد النص قبل الترميز: إزالة التشكيل + توحيد المسافات
TASHKEEL = "".join(chr(c) for c in list(range(0x064B, 0x0660)) + [0x0670])


def normalize_ar(text):
    text = text.translate(str.maketrans("", "", TASHKEEL))
    return re.sub(r"\s+", " ", text).strip()


MODEL_NAME = "omarelshehy/Arabic-Retrieval-v1.0"

embedding_model = SentenceTransformer(MODEL_NAME, device="cpu")

print("Model:", MODEL_NAME)
print("Embedding dimension:", embedding_model.get_embedding_dimension())

Model: omarelshehy/Arabic-Retrieval-v1.0
Embedding dimension: 768


In [3]:
texts = [normalize_ar(doc["content"]) for doc in documents]

all_embeddings = embedding_model.encode_document(
    texts,
    normalize_embeddings=True,
    batch_size=8,
    show_progress_bar=True,
)

print("Embeddings shape:", all_embeddings.shape)
print("Contains NaN:", bool(np.isnan(all_embeddings).any()))
print("Contains Inf:", bool(np.isinf(all_embeddings).any()))

Embeddings shape: (116, 768)
Contains NaN: False
Contains Inf: False


In [4]:
embeddings_path = "../data/hadith_embeddings.npy"

np.save(embeddings_path, all_embeddings)

print(f"Saved embeddings to: {embeddings_path}")

Saved embeddings to: ../data/hadith_embeddings.npy


In [5]:
test_queries = [
    ("ما الحديث الذي يتحدث عن النيات؟", 1),
    ("ما هي مراتب الدين؟", 2),
    ("ما هي مراحل خلق الإنسان؟", 4),
    ("ما حكم الأمور المشتبهة؟", 6),
    ("ما المقصود بالنصيحة في الدين؟", 7),
]

correct = 0
for query, expected in test_queries:
    qv = embedding_model.encode_query(normalize_ar(query), normalize_embeddings=True)
    scores = all_embeddings @ qv
    best = int(np.argmax(scores))
    predicted = documents[best]["metadata"]["hadith_number"]
    ok = predicted == expected
    correct += ok
    print(f"Query: {query}")
    print(f"  Expected: {expected} | Predicted: {predicted} | Score: {scores[best]:.4f} | {'PASS' if ok else 'FAIL'}")

print(f"\nAccuracy: {correct}/{len(test_queries)}")

Query: ما الحديث الذي يتحدث عن النيات؟
  Expected: 1 | Predicted: 1 | Score: 0.4291 | PASS
Query: ما هي مراتب الدين؟
  Expected: 2 | Predicted: 2 | Score: 0.6009 | PASS
Query: ما هي مراحل خلق الإنسان؟
  Expected: 4 | Predicted: 4 | Score: 0.6215 | PASS
Query: ما حكم الأمور المشتبهة؟
  Expected: 6 | Predicted: 6 | Score: 0.4061 | PASS
Query: ما المقصود بالنصيحة في الدين؟
  Expected: 7 | Predicted: 7 | Score: 0.6845 | PASS

Accuracy: 5/5


In [6]:
# اختبار ثان: أسئلة أصعب (رواة / معاني / صياغات عامية) على كل الوثائق
test_queries_2 = [
    ("من هو راوي حديث مراتب الدين؟", 2),
    ("أي الحديث اللي بيتكلم عن مراتب الدين؟", 2),
    ("ما هي أركان الإسلام الخمسة؟", 3),
    ("حديث النهي عن الغضب", 16),
    ("ما جزاء معاداة أولياء الله؟", 38),
    ("حديث لا ضرر ولا ضرار", 32),
    ("ما هي علامات الساعة في حديث جبريل؟", 2),
    ("من راوي حديث إنما الأعمال بالنيات؟", 1),
    ("حديث الاستقامة", 21),
    ("ما حكم من أحدث في الدين ما ليس منه؟", 5),
]

def doc_num(d):
    md = d["metadata"]
    if "hadith_number" in md:
        return md["hadith_number"]
    nums = md.get("hadith_numbers") or []
    return nums[0] if nums else None

correct = 0
for query, expected in test_queries_2:
    qv = embedding_model.encode_query(normalize_ar(query), normalize_embeddings=True)
    scores = all_embeddings @ qv
    order = np.argsort(scores)[::-1][:3]
    top1 = doc_num(documents[int(order[0])])
    ok = top1 == expected
    correct += ok
    top3 = [(doc_num(documents[i]), round(float(scores[i]), 4)) for i in order]
    print(f"Query: {query}")
    print(f"  Expected: {expected} | Top3: {top3} | {'PASS' if ok else 'FAIL'}")

print(f"\nAccuracy: {correct}/{len(test_queries_2)}")


Query: من هو راوي حديث مراتب الدين؟
  Expected: 2 | Top3: [(24, 0.3753), (11, 0.3318), (30, 0.3145)] | FAIL
Query: أي الحديث اللي بيتكلم عن مراتب الدين؟
  Expected: 2 | Top3: [(2, 0.4714), (7, 0.377), (4, 0.2888)] | PASS
Query: ما هي أركان الإسلام الخمسة؟
  Expected: 3 | Top3: [(3, 0.7788), (3, 0.6998), (2, 0.5134)] | PASS
Query: حديث النهي عن الغضب
  Expected: 16 | Top3: [(16, 0.7578), (16, 0.6777), (9, 0.5638)] | PASS
Query: ما جزاء معاداة أولياء الله؟
  Expected: 38 | Top3: [(38, 0.6347), (38, 0.6193), (35, 0.4129)] | PASS
Query: حديث لا ضرر ولا ضرار
  Expected: 32 | Top3: [(32, 0.7382), (32, 0.7262), (27, 0.4107)] | PASS
Query: ما هي علامات الساعة في حديث جبريل؟
  Expected: 2 | Top3: [(4, 0.3674), (19, 0.3565), (20, 0.3499)] | FAIL
Query: من راوي حديث إنما الأعمال بالنيات؟
  Expected: 1 | Top3: [(1, 0.6262), (1, 0.5947), (11, 0.5259)] | PASS
Query: حديث الاستقامة
  Expected: 21 | Top3: [(21, 0.551), (21, 0.5009), (12, 0.4759)] | PASS
Query: ما حكم من أحدث في الدين ما ليس منه؟
  Exp